# Etapa 3 — Rodando LOCAL (PC com GTX 1650 Super, 16 GB RAM)

Pré-requisitos (no terminal do VS Code, com o `.venv` ativado):
1. `python -m venv .venv` e ativar (`.venv\\Scripts\\activate` no Windows)
2. PyTorch com CUDA — pegue o comando em https://pytorch.org/get-started/locally (para a 1650 Super, a build `cu124` funciona)
3. `pip install -r requirements.txt`

Ajustes pro seu hardware já embutidos: **AMP ligado** (corta VRAM) e **batch pequeno** (cabe em 4 GB).

## 0. Apontar para a raiz do projeto

In [ ]:
import os, sys
# rode este notebook de dentro da pasta do projeto (onde está a pasta src/)
ROOT = os.getcwd()
if ROOT not in sys.path: sys.path.insert(0, ROOT)
import torch
print('CUDA disponivel?', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Baixar o dataset (uma vez)
O arquivo `pathmnist_224.npz` (~3 GB) é grande. **O jeito mais confiável é baixar pelo navegador** (ele retoma se cair), neste link:

https://zenodo.org/records/10519652/files/pathmnist_224.npz?download=1

Salve em `data/pathmnist_224.npz` dentro do projeto. Se preferir baixar por código, use a célula abaixo (com retomada e tentativas).

In [ ]:
import os, urllib.request
os.makedirs('data', exist_ok=True)
NPZ = 'data/pathmnist_224.npz'
URL = 'https://zenodo.org/records/10519652/files/pathmnist_224.npz?download=1'

def baixar_com_retomada(url, dest, tentativas=30):
    for t in range(tentativas):
        pos = os.path.getsize(dest) if os.path.exists(dest) else 0
        req = urllib.request.Request(url, headers={'Range': f'bytes={pos}-'})
        try:
            with urllib.request.urlopen(req, timeout=60) as r, open(dest, 'ab') as f:
                while True:
                    chunk = r.read(1<<20)
                    if not chunk: break
                    f.write(chunk)
            print('download completo:', os.path.getsize(dest)//1024//1024, 'MB'); return
        except Exception as e:
            print(f'tentativa {t+1} caiu ({e}); retomando...')
    raise RuntimeError('download falhou apos varias tentativas')

# baixar_com_retomada(URL, NPZ)   # descomente para baixar por codigo

## 2. Converter para formato econômico de RAM (uma vez)
Transforma o `.npz` em arrays `.npy` mmapeáveis. Gasta ~16 GB de disco, mas resolve o estouro de RAM. Pula sozinho se já tiver convertido.

In [ ]:
from src.utils.data_mmap import convert
DATA_DIR = 'data/pathmnist224'
if not os.path.exists(os.path.join(DATA_DIR, 'train_images.npy')):
    convert(NPZ, DATA_DIR)
else:
    print('ja convertido:', DATA_DIR)

## 3. Loaders econômicos de RAM
`batch_size=8` é seguro para 4 GB de VRAM com AMP. Suba para 16 nos modelos leves (EfficientNet, CNN autoral) se sobrar memória.

In [ ]:
from src.utils.seeds import set_seed
from src.utils.data_mmap import make_loaders_local
set_seed(42)
train_loader, val_loader, test_loader = make_loaders_local(
    DATA_DIR, batch_size=8, num_workers=4, augment=True)
print('batches treino:', len(train_loader))

## 4. Varredura otimizador × learning rate
No ResNet50 (feature extraction = mais leve e rápido) para escolher a config.

In [ ]:
from src.etapa3_cnn_vit.run_experiments import sweep_optim_lr
melhor = sweep_optim_lr('resnet50', 'feature_extraction',
                        train_loader, val_loader, epochs=3, pretrained=True)
print('Melhor:', melhor)

## 5. Comparação dos modelos
⚠️ Pesado na 1650 Super. Comece com `epochs` baixo. **VGG16 e ViT em fine-tuning** são os mais lentos/pesados — se der OOM de VRAM, rode com `batch_size=4` (recrie os loaders na célula 3). Pode levar várias horas: considere rodar à noite.

In [ ]:
from src.etapa3_cnn_vit.run_experiments import compare_models
resultados = compare_models(train_loader, val_loader,
                            optimizer=melhor['otimizador'], lr=melhor['lr'],
                            epochs=5, pretrained=True)
import pandas as pd; pd.DataFrame(resultados)

## 6. (Só no fim do projeto) Avaliação no TESTE
O conjunto de teste só pode ser usado **uma vez**, sobre o melhor modelo final (após a Etapa 5).